# Lab 14: Stability, Ranking, and Iteration

## Repeated matrix action as a way to see the future

In this lab, we study the equation

$$x_{k+1}=Ax_k.$$

This looks simple, but it describes many important systems:

- population growth,
- movement between states,
- ranking webpages,
- smoothing signals and images,
- diffusion on networks,
- and numerical algorithms such as power iteration.

The goal is not only to compute $Ax$. The goal is to understand what happens after many steps.

We will use Python to see eigenvalues, stable directions, convergence, ranking, and high-dimensional behavior.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. Iterating a simple diagonal system

A diagonal matrix is the easiest place to see the idea.

Let

$$A=egin{bmatrix}1.2&0\0&0.55\end{bmatrix}.$$

The first coordinate grows. The second coordinate decays.

In [ ]:
A = np.array([[1.2, 0.0],
              [0.0, 0.55]])
x = np.array([1.0, 2.0])

history = [x.copy()]
for k in range(25):
    x = A @ x
    history.append(x.copy())

history = np.array(history)

plt.figure(figsize=(7, 5))
plt.plot(history[:, 0], label='first coordinate')
plt.plot(history[:, 1], label='second coordinate')
plt.axhline(0, linewidth=1)
plt.xlabel('iteration k')
plt.ylabel('coordinate value')
plt.title('One direction grows while another direction decays')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(history[:,0], history[:,1], marker='o')
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.xlabel('x-coordinate')
plt.ylabel('y-coordinate')
plt.title('Path of the state vector under repeated multiplication')
plt.axis('equal')
plt.show()

### Student task

Change the numbers $1.2$ and $0.55$.

Try to create:

1. a system that decays to zero,
2. a system that grows in both directions,
3. a system that grows in one direction and flips sign in the other.

## 2. Eigenvectors explain the movie

For a general matrix, the coordinate axes may not be the special directions.

The special directions are eigenvectors.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])

eigvals, eigvecs = np.linalg.eig(A)
print('Eigenvalues:')
print(eigvals)
print('
Eigenvectors as columns:')
print(eigvecs)

In [ ]:
# Plot the vector field x -> A x for points on the unit circle
angles = np.linspace(0, 2*np.pi, 80)
points = np.c_[np.cos(angles), np.sin(angles)]
images = points @ A.T

plt.figure(figsize=(7, 7))
plt.plot(points[:,0], points[:,1], label='unit circle')
plt.plot(images[:,0], images[:,1], label='image under A')

for j in range(2):
    v = eigvecs[:, j]
    plt.arrow(0, 0, v[0], v[1], head_width=0.05, length_includes_head=True)
    plt.arrow(0, 0, -v[0], -v[1], head_width=0.05, length_includes_head=True)

plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.axis('equal')
plt.legend()
plt.title('Eigenvectors are directions preserved by the transformation')
plt.show()

## 3. Power iteration

Power iteration repeatedly multiplies by $A$ and normalizes.

$$x_{k+1}=rac{Ax_k}{\|Ax_k\|}.$$

It often finds the dominant eigenvector.

In [ ]:
def power_iteration(A, x0=None, steps=20):
    n = A.shape[0]
    if x0 is None:
        x = np.random.default_rng(0).normal(size=n)
    else:
        x = np.array(x0, dtype=float)
    x = x / np.linalg.norm(x)
    hist = [x.copy()]
    rayleigh = []
    for _ in range(steps):
        y = A @ x
        x = y / np.linalg.norm(y)
        hist.append(x.copy())
        rayleigh.append(x @ A @ x)
    return np.array(hist), np.array(rayleigh)

A = np.array([[2, 1], [1, 2]], dtype=float)
hist, rays = power_iteration(A, x0=[1, 0], steps=15)
print(hist[-1])
print('Rayleigh quotient estimates:', np.round(rays[-5:], 5))

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(rays, marker='o')
plt.axhline(max(np.linalg.eigvals(A)), linestyle='--')
plt.xlabel('iteration')
plt.ylabel('Rayleigh quotient')
plt.title('Power iteration estimates the dominant eigenvalue')
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(hist[:,0], hist[:,1], marker='o')
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.axis('equal')
plt.title('Direction converges to the dominant eigenvector')
plt.show()

## 4. Convergence speed and spectral gap

Power iteration is faster when the largest eigenvalue is clearly separated from the second largest eigenvalue.

The rough speed factor is

$$\left|rac{\lambda_2}{\lambda_1}ight|.$$

In [ ]:
def compare_power_speed(lam1, lam2, steps=30):
    Q = np.array([[np.cos(0.7), -np.sin(0.7)],
                  [np.sin(0.7),  np.cos(0.7)]])
    D = np.diag([lam1, lam2])
    A = Q @ D @ Q.T
    v_dom = Q[:,0]
    hist, _ = power_iteration(A, x0=[1, 2], steps=steps)
    errors = []
    for x in hist:
        # direction error ignores sign
        errors.append(min(np.linalg.norm(x-v_dom), np.linalg.norm(x+v_dom)))
    return np.array(errors)

fast = compare_power_speed(5.0, 1.0)
slow = compare_power_speed(5.0, 4.7)

plt.figure(figsize=(7,5))
plt.semilogy(fast, marker='o', label='large spectral gap: 5 vs 1')
plt.semilogy(slow, marker='o', label='small spectral gap: 5 vs 4.7')
plt.xlabel('iteration')
plt.ylabel('direction error, log scale')
plt.title('Spectral gap controls convergence speed')
plt.legend()
plt.show()

## 5. Markov chains: probability vectors under iteration

A column-stochastic matrix has nonnegative entries and columns that sum to $1$.

If $p_k$ is a probability vector, then $p_{k+1}=Pp_k$ is the next probability distribution.

In [ ]:
P = np.array([[0.8, 0.3],
              [0.2, 0.7]])

print('Column sums:', P.sum(axis=0))

p = np.array([1.0, 0.0])
history = [p.copy()]
for k in range(30):
    p = P @ p
    history.append(p.copy())
history = np.array(history)

plt.figure(figsize=(7,5))
plt.plot(history[:,0], marker='o', label='Sunny probability')
plt.plot(history[:,1], marker='o', label='Rainy probability')
plt.xlabel('day')
plt.ylabel('probability')
plt.title('Weather Markov chain converges to a stationary distribution')
plt.legend()
plt.show()

print('Final distribution:', history[-1])

In [ ]:
# Solve stationary distribution by eigenvectors
vals, vecs = np.linalg.eig(P)
idx = np.argmin(np.abs(vals - 1))
pi = np.real(vecs[:, idx])
pi = pi / pi.sum()
print('Stationary distribution:', pi)
print('Check P @ pi:', P @ pi)

## 6. Ranking from a network

A ranking vector is stable when importance flowing through links reproduces the same vector.

$$r = Mr.$$

This is an eigenvector equation with eigenvalue $1$.

In [ ]:
# Column j tells where importance from page j goes.
M = np.array([
    [0,   1/2, 1/2, 0],
    [1/2, 0,   1/2, 1/3],
    [1/2, 1/2, 0,   1/3],
    [0,   0,   0,   1/3]
], dtype=float)

print('Column sums:', M.sum(axis=0))

r = np.ones(4) / 4
hist = [r.copy()]
for k in range(50):
    r = M @ r
    hist.append(r.copy())
hist = np.array(hist)

print('Ranking:', r / r.sum())

In [ ]:
plt.figure(figsize=(8,5))
for i in range(4):
    plt.plot(hist[:,i], label=f'page {i+1}')
plt.xlabel('iteration')
plt.ylabel('importance')
plt.title('Importance scores under repeated link-following')
plt.legend()
plt.show()

## 7. PageRank-style damping

Damping means that a user usually follows a link, but sometimes jumps randomly to any page.

$$G=lpha M+(1-lpha)rac{1}{n}\mathbf{1}\mathbf{1}^T.$$

In [ ]:
alpha = 0.85
n = M.shape[0]
G = alpha * M + (1-alpha) * np.ones((n,n)) / n

r = np.ones(n) / n
hist = [r.copy()]
for k in range(80):
    r = G @ r
    hist.append(r.copy())
hist = np.array(hist)

print('Damped ranking:', np.round(r / r.sum(), 4))
print('Column sums:', np.round(G.sum(axis=0), 6))

In [ ]:
plt.figure(figsize=(8,5))
for i in range(n):
    plt.plot(hist[:,i], label=f'page {i+1}')
plt.xlabel('iteration')
plt.ylabel('importance')
plt.title('Damped ranking converges smoothly')
plt.legend()
plt.show()

## 8. Draw the network

We can visualize the pages as points on a circle and draw arrows for links.

In [ ]:
def draw_network(M):
    n = M.shape[0]
    theta = np.linspace(0, 2*np.pi, n, endpoint=False)
    pos = np.c_[np.cos(theta), np.sin(theta)]
    plt.figure(figsize=(6,6))
    plt.scatter(pos[:,0], pos[:,1], s=900)
    for i, (x,y) in enumerate(pos):
        plt.text(x, y, str(i+1), ha='center', va='center', fontsize=14)
    for j in range(n):
        for i in range(n):
            if M[i,j] > 0 and i != j:
                start = pos[j]
                end = pos[i]
                vec = end - start
                plt.arrow(start[0]*0.85, start[1]*0.85,
                          vec[0]*0.65, vec[1]*0.65,
                          head_width=0.06, length_includes_head=True, alpha=0.7)
    plt.axis('equal')
    plt.axis('off')
    plt.title('Network used for ranking')
    plt.show()

draw_network(M)

## 9. Iteration as smoothing and diffusion

A smoothing matrix averages neighboring entries.

Repeated smoothing removes rough details.

In [ ]:
def smoothing_matrix(n, self_weight=0.5):
    S = np.zeros((n,n))
    for i in range(n):
        S[i,i] = self_weight
        if i > 0:
            S[i,i-1] += (1-self_weight)/2
        else:
            S[i,i] += (1-self_weight)/2
        if i < n-1:
            S[i,i+1] += (1-self_weight)/2
        else:
            S[i,i] += (1-self_weight)/2
    return S

n = 80
S = smoothing_matrix(n, self_weight=0.55)
x = np.zeros(n)
x[15:25] = 1
x[45:50] = 1.4
x += 0.25*np.random.default_rng(1).normal(size=n)

signals = [x.copy()]
for k in range(30):
    x = S @ x
    if k in [0, 1, 3, 7, 15, 29]:
        signals.append(x.copy())

plt.figure(figsize=(9,5))
for s in signals:
    plt.plot(s, alpha=0.8)
plt.title('Repeated smoothing removes rough detail')
plt.xlabel('position')
plt.ylabel('signal value')
plt.show()

## 10. Image diffusion

An image can be treated as a matrix of pixel values. Repeated local averaging blurs the image.

In [ ]:
def blur_image(img, steps=1):
    out = img.copy()
    for _ in range(steps):
        padded = np.pad(out, 1, mode='edge')
        out = (padded[1:-1,1:-1] + padded[:-2,1:-1] + padded[2:,1:-1] + padded[1:-1,:-2] + padded[1:-1,2:]) / 5
    return out

# Create a synthetic image: two bright squares and diagonal stripe
img = np.zeros((60,60))
img[10:25, 12:28] = 1.0
img[34:50, 35:52] = 0.8
for i in range(60):
    img[i, max(0,i-2):min(60,i+2)] = 0.5
img += 0.15*np.random.default_rng(2).normal(size=img.shape)
img = np.clip(img, 0, 1)

steps_list = [0, 1, 3, 8, 20]
for steps in steps_list:
    plt.figure(figsize=(4,4))
    plt.imshow(blur_image(img, steps), cmap='gray', vmin=0, vmax=1)
    plt.title(f'Blur steps = {steps}')
    plt.axis('off')
    plt.show()

## 11. High-dimensional power iteration

Power iteration is useful because many real matrices are too large for complete eigenvalue decomposition.

Here we build a random symmetric matrix with a deliberately strong hidden direction.

In [ ]:
rng = np.random.default_rng(3)
n = 100
v_true = rng.normal(size=n)
v_true = v_true / np.linalg.norm(v_true)

# Matrix with one strong rank-one direction plus noise
A = 6 * np.outer(v_true, v_true) + rng.normal(scale=0.25, size=(n,n))
A = (A + A.T) / 2

hist, rays = power_iteration(A, steps=40)
cosines = np.abs(hist @ v_true)

plt.figure(figsize=(7,5))
plt.plot(cosines, marker='o')
plt.xlabel('iteration')
plt.ylabel('absolute cosine with hidden direction')
plt.title('Power iteration finds a hidden high-dimensional direction')
plt.ylim(0, 1.05)
plt.show()

## 12. Reflection questions

1. In your own words, why do eigenvectors matter more after many iterations than after one multiplication?
2. What does a stationary distribution mean in a Markov chain?
3. Why does PageRank-style damping help convergence?
4. Why does a small spectral gap make power iteration slow?
5. How is image smoothing an eigenvector story?

## 13. Extension project

Create your own network with 6 to 10 nodes.

1. Build its column-stochastic link matrix.
2. Compute the ranking by iteration.
3. Add damping and compute the damped ranking.
4. Visualize the convergence curves.
5. Explain which node ranks highest and why.